# Dynamic Graph CNN (DGCNN) for Point Cloud Classification

Point Cloud Classification on ModelNet / MedShapeNet: 3D point cloud classification with dynamic k-NN graphs and EdgeConv. This notebook implements the approach with `DynamicEdgeConv` inside a `K3DGCNN` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DynamicEdgeConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Dynamic Graph CNN (DGCNN) for Point Cloud Classification"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. DGCNN Model Definition
class K3DGCNN(keras.Model):
    def __init__(self, out_channels, k=20):
        super().__init__()
        self.k = k
        self.conv1 = k3_layers.DynamicEdgeConv(
            keras.Sequential([layers.Dense(64, activation="relu"), layers.Dense(64)]),
            k=k,
        )
        self.conv2 = k3_layers.DynamicEdgeConv(
            keras.Sequential([layers.Dense(64, activation="relu"), layers.Dense(64)]),
            k=k,
        )
        self.lin1 = layers.Dense(512, activation="relu")
        self.lin2 = layers.Dense(out_channels)

    def call(self, pos, batch=None):
        x1 = self.conv1(pos, batch=batch)
        x2 = self.conv2(x1, batch=batch)
        out = ops.concatenate([x1, x2], axis=-1)
        out = k3_layers.global_max_pool(out, batch)
        out = self.lin1(out)
        return self.lin2(out)

k3_model = K3DGCNN(out_channels=10, k=16)

# 2. Sample Forward Pass & Verification
batch_size, num_points = 4, 128
dummy_pos = keras.random.normal((batch_size * num_points, 3))
dummy_batch = ops.repeat(ops.arange(batch_size, dtype="int64"), num_points)

out = k3_model(dummy_pos, batch=dummy_batch)
print(f"Forward pass completed! Output shape: {out.shape} (Expected: ({batch_size}, 10))")

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)
print("Model compiled successfully!")

print("\n✓ K3-Node DGCNN execution completed successfully!")